# PETadex clustering: substring &amp; near-substring scan

**Goal.** Find and group amino-acid sequences in the joined PETadex dataset that are variants of one another, so a reviewer can spot duplicates, engineered mutants, and truncations/extensions of the same underlying enzyme.

**Two passes, same pipeline.**

| Pass | Script | Match rule | Catches | Sheet written |
|------|--------|-----------|---------|---------------|
| Exact | `substring_scan.py` | shorter is a *contiguous exact substring* of longer | terminal add/truncation, exact internal containment | `substring-scan` |
| Near | `substring_scan_extended.py` | difflib matching-block coverage ≥ `min_coverage` | the above **plus** point substitutions &amp; internal indels | `near-substring-scan` |

Both passes share one pipeline: load unique-sequence records → union-find grouping over a pairwise match predicate → label each group (shortest = `core`, longer members = `superset`) → write a styled sheet. Only the **match predicate** and **position labeller** differ between passes.

All settings live in `config.yaml`.

## 0. Setup &amp; configuration

Requirements: `pandas`, `openpyxl`, and (optionally) `PyYAML` for the config file. Run this notebook from the `petadex-clustering/` folder so relative paths resolve.

In [ ]:
import pandas as pd

import substring_scan as base
import substring_scan_extended as ext

cfg = base.load_config()
cfg

## 1. Load the unique-sequence records

`load_records` reads the joined sheet, normalizes each sequence (strip whitespace, uppercase) into a match `key`, coalesces the enzyme-name and accession columns, and de-duplicates on the normalized key.

In [ ]:
df = base.load_records(cfg['workbook'], cfg['joined_sheet'])
print(f"{len(df)} unique sequences; lengths {df['len'].min()}–{df['len'].max()} aa")
df[['enzyme_name', 'accessions', 'source', 'len']].head()

## 2. Exact-substring pass

The pairwise test is `exact_substring_match` = `short_key in long_key`. This is the conservative, no-false-positives pass. `run()` writes the `substring-scan` sheet and returns the flat result table.

In [ ]:
exact_scan = base.run(config=cfg)   # match=exact_substring_match by default
exact_scan.head(12)

## 3. Near-substring pass

The relaxed predicate (`make_near_match`) also accepts a pair when difflib matching blocks cover at least `min_coverage` of the shorter sequence — so a few point substitutions or internal indels no longer break the grouping. Exact containments keep their precise `prefix`/`suffix`/`internal` label; genuinely near (non-exact) matches are labelled `near`.

This writes a **separate** `near-substring-scan` sheet, so the exact results above are left untouched.

In [ ]:
near_match = ext.make_near_match(cfg['near']['min_coverage'])
near_scan = base.run(match=near_match, position_fn=ext.near_position,
                     sheet_name=cfg['near']['sheet_name'], config=cfg)
near_scan.head(12)

## 4. Compare the two passes

The near pass should be a *superset* of the exact pass: every exact grouping still holds, plus additional near-variant members. The rows the near pass adds — especially those labelled `near` — are the interesting new candidates to review.

In [ ]:
summary = pd.DataFrame([
    {'pass': 'exact', 'groups': exact_scan['group'].nunique(),
     'sequences_involved': len(exact_scan),
     'supersets': int((exact_scan['role'] == 'superset').sum())},
    {'pass': 'near', 'groups': near_scan['group'].nunique(),
     'sequences_involved': len(near_scan),
     'supersets': int((near_scan['role'] == 'superset').sum())},
])
summary

In [ ]:
# Rows found only by the relaxed pass (the near-variant candidates).
near_only = near_scan[near_scan['core_position'] == 'near']
print(f"{len(near_only)} 'near' rows the exact pass would miss")
near_only[['group', 'role', 'enzyme_name', 'accessions', 'length_aa', 'delta_aa']].head(20)

## 5. Sensitivity to the threshold (optional)

`min_coverage` is the one knob that changes the near pass. Sweep it to see how many groups form at each stringency before committing a value to `config.yaml`. Higher = stricter (fewer, more confident groups).

In [ ]:
rows = []
for thr in [0.85, 0.90, 0.92, 0.95, 0.98]:
    groups = base.build_groups(df, ext.make_near_match(thr))
    rows.append({'min_coverage': thr, 'groups': len(groups),
                 'sequences_involved': sum(len(g) for g in groups)})
pd.DataFrame(rows)

## Caveats &amp; next steps

- The near pass is a **difflib heuristic**, not a biological alignment. For publication-grade calls, swap `make_near_match` for a Biopython pairwise aligner and score on percent identity / gap penalties.
- Grouping is transitive (union-find): if A–B and B–C match, A, B, C land in one group even if A and C are not directly similar. Inspect large groups.
- The near pass is O(n²) difflib comparisons — fine at this dataset size, but revisit if the unique-sequence count grows a lot.
- The `change_or_comment` and `most_wildtype` columns are intentionally left blank (yellow-filled) for manual curation in Excel.